# **Import Libraries**

In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image, ImageTk
import logging

# **Logging**

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# **Load Trained Model**

In [3]:
try:
    model = load_model('rice_cnn_model.h5')
except Exception as e:
    logging.error(f"Error loading model: {e}")
    exit()

2025-03-24 22:16:41,955 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


# **Load Dictionary**

In [4]:
labels_dict = {
    0: '28',
    1: '29',
    2: 'Basmati',
    3: 'Chinipata',
    4: 'Miniket'
}

# **GUI**

In [5]:
class RiceDetectorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Rice Detector (CNN)")
        self.root.geometry("800x600")

        self.label = tk.Label(root, text="Select an image to detect rice type", font=("Arial", 14))
        self.label.pack(pady=10)

        self.select_button = tk.Button(root, text="Select Image", command=self.select_image, font=("Arial", 12))
        self.select_button.pack(pady=10)

        self.image_label = tk.Label(root)
        self.image_label.pack(pady=10)

        self.result_label = tk.Label(root, text="", font=("Arial", 12), fg="blue")
        self.result_label.pack(pady=10)

    def select_image(self):
        file_path = filedialog.askopenfilename(
            title="Select Rice Image",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.bmp")]
        )
        
        if not file_path:
            return

        image = cv2.imread(file_path, cv2.IMREAD_COLOR)
        if image is None:
            messagebox.showerror("Error", "Could not load the image!")
            return

        # Resize and preprocess for CNN
        cnn_input = cv2.resize(image, (224, 224))
        cnn_input = cnn_input / 255.0
        cnn_input = np.expand_dims(cnn_input, axis=0)

        # Display image
        display_image = cv2.resize(image, (660, 380))
        
        # Predict with CNN
        probs = model.predict(cnn_input)[0]
        max_prob = np.max(probs)
        prediction = np.argmax(probs)
        print(f"Prediction index: {prediction}, Confidence: {max_prob}")
        print(f"Probabilities: {probs}")

        # Result text
        if max_prob >= 0.5:  # Confidence থ্রেশহোল্ড
            predicted_value = labels_dict.get(prediction, "Unknown")
            result_text = f"Detected: {predicted_value} (Confidence: {max_prob:.2f})"
            cv2.rectangle(display_image, (0, 0), (660, 380), (0, 255, 0), 3)
        else:
            result_text = "No Rice Detected"

        # Convert image for Tkinter
        image_rgb = cv2.cvtColor(display_image, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(image_rgb)
        imgtk = ImageTk.PhotoImage(image=img)

        # Update GUI
        self.image_label.config(image=imgtk)
        self.image_label.image = imgtk
        self.result_label.config(text=result_text)

# **Run GUI**

In [ ]:
if __name__ == "__main__":
    root = tk.Tk()
    app = RiceDetectorApp(root)
    root.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step
Prediction index: 4, Confidence: 0.8125080466270447
Probabilities: [1.8724032e-01 7.0095517e-08 2.0835815e-10 2.5153873e-04 8.1250805e-01]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction index: 0, Confidence: 0.5695486068725586
Probabilities: [5.6954861e-01 4.3080105e-05 8.2801073e-08 1.7391579e-02 4.1301665e-01]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction index: 3, Confidence: 0.9435163140296936
Probabilities: [4.0489299e-06 5.6479238e-02 1.9599167e-14 9.4351631e-01 4.2803680e-07]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Prediction index: 1, Confidence: 0.9903312921524048
Probabilities: [2.8153455e-03 9.9033129e-01 5.1075572e-06 6.8297018e-03 1.8518724e-05]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Prediction index: 1, Confidence: 0.9997256398200989
Probabilities: [1.4078778e-07 9.9972564e-01 4.8788031e-12 2.7425087e-04 9.8281964e-12]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Prediction index: 2, Confidence: 0.9999998807907104
Probabilities: [1.488805